# Ch 22. DPO와 최신 정렬 기법 — 해설

> 이 노트북은 다섯 연습문제의 재현 가능한 해설을 제공한다. 모든 출력은 비워 두었으며 코드 셀은 위에서 아래로 독립적인 CPU 환경에서 실행된다.


## 문제 1

**문제**: DPO 손실에서 $\beta = 0.01, 0.1, 1.0$을 비교하고 효과를 설명하라.

### 해설

DPO logit은 $\beta[(\log\pi_w-\log\pi_{ref,w})-(\log\pi_l-\log\pi_{ref,l})]$이다. 같은 margin에서 $\beta$가 클수록 sigmoid가 포화되어 더 날카로운 선호와 작은 유효 그래디언트를 만들 수 있다. 아래에서 손실과 그래디언트를 함께 비교한다.


In [ ]:
import numpy as np

margin=2.0; results={}
for beta in (.01,.1,1.0):
    logit=beta*margin; loss=float(np.logaddexp(0,-logit)); gradient=float(-beta/(1+np.exp(logit)))
    results[beta]={"loss":loss,"d_loss_d_margin":gradient}
assert results[1.0]["loss"] < results[.1]["loss"] < results[.01]["loss"]
print({str(k):{m:round(v,6) for m,v in r.items()} for k,r in results.items()})


## 문제 2

**문제**: Bradley-Terry 모델이 "선호 쌍"을 어떻게 모델링하는지 설명하라.

### 해설

Bradley–Terry는 두 응답의 잠재 효용 차이를 로짓으로 사용해 $P(y_w\succ y_l)=\sigma(r_w-r_l)$로 둔다. 두 확률의 합은 1이며 공통 상수를 두 점수에 더해도 확률은 변하지 않는다.


In [ ]:
import numpy as np

chosen=np.array([-1.0,0.0,2.0]); rejected=np.array([0.5,0.0,-1.0]); gaps=chosen-rejected
prob=1/(1+np.exp(-gaps)); reverse=1/(1+np.exp(gaps))
assert np.allclose(prob+reverse,1) and np.isclose(prob[1],.5)
print([{"utility_gap":float(g),"P(chosen preferred)":round(float(p),4)} for g,p in zip(gaps,prob)])


## 문제 3

**문제**: DPO와 PPO의 메모리 사용량을 비교하라.

### 해설

PPO는 일반적으로 정책·참조·보상·가치의 네 모델과 rollout 버퍼가 필요하다. DPO는 정책·참조 두 모델의 chosen/rejected 순전파가 핵심이다. 정확한 바이트 수는 옵티마이저, 양자화, 체크포인팅에 의존하지만 동등 정밀도에서 모델 상태의 하한은 대략 절반이다.


In [ ]:
# Explicit memory ledger in model-size units; batch buffers are separately measured assumptions.
ppo={"policy":1.0,"reference":1.0,"reward":1.0,"value":1.0,"rollouts":0.30}
dpo={"policy":1.0,"reference":1.0,"paired_batch":0.10}
totals={"PPO":sum(ppo.values()),"DPO":sum(dpo.values())}
assert totals["DPO"] < totals["PPO"] and set(ppo)-set(dpo)=={"reward","value","rollouts"}
print({"components":{"PPO":ppo,"DPO":dpo},"totals":totals,"DPO_fraction":round(totals["DPO"]/totals["PPO"],3)})


## 문제 4

**문제**: KTO가 왜 선호 쌍 없이 학습 가능한지 설명하라.

### 해설

KTO는 각 응답에 desirable/undesirable 라벨을 부여하고 참조 대비 로그확률 비율의 가치 함수를 전망이론식 비대칭 손실로 최적화한다. 한 응답의 라벨만으로 항을 만들 수 있으므로 같은 프롬프트의 명시적 짝이 필요 없다.


In [ ]:
import numpy as np

# Independent labeled examples contribute without constructing chosen/rejected pairs.
log_ratio=np.array([.7,-.4,.2,-.8]); desirable=np.array([1,0,1,0],dtype=bool)
signed_value=np.where(desirable,log_ratio,-log_ratio)
loss=np.logaddexp(0,-signed_value)
assert loss.shape==(4,) and loss[0] < loss[2] and loss[3] < loss[1]
print([{"label":"desirable" if y else "undesirable","log_ratio":float(r),"loss":round(float(l),4)}
       for y,r,l in zip(desirable,log_ratio,loss)])


## 문제 5

**문제**: 보상-정책 관계식 $r = \beta \log \frac{\pi^*}{\pi_{\text{ref}}} + \beta \log Z$를 유도하라.

### 해설

KL 정규화 목적을 각 $x$에 대해 라그랑주 승수로 풀면 $\pi^*(y|x)\propto\pi_{ref}(y|x)e^{r(x,y)/\beta}$다. 정규화 상수 $Z(x)$를 넣고 로그를 취해 정리하면 요구한 식이며, 응답 쌍의 차에서는 $\beta\log Z(x)$가 상쇄된다.


In [ ]:
import numpy as np

reward=np.array([1.2,-.3,.7]); beta=.2; reference=np.array([.2,.5,.3])
unnormalized=reference*np.exp(reward/beta); policy=unnormalized/unnormalized.sum(); Z=unnormalized.sum()
reconstructed=beta*np.log(policy/reference)+beta*np.log(Z)
max_error=float(np.max(np.abs(reconstructed-reward)))
assert np.allclose(reconstructed,reward,atol=1e-12) and np.isclose(policy.sum(),1)
print({"reference":reference.tolist(),"optimal_policy":policy.round(6).tolist(),"partition_Z":round(float(Z),6),"max_reconstruction_error":max_error})
